# Phase 2 — Legal Named Entity Recognition (L-NER)  ·  v2.1 (score-focused rewrite + span head + error analysis)
**Kaggle notebook · Phase-1 Legal LM transfer · exact character-span evaluation**

v1 scored **0.4075** on the leaderboard (validation 0.373: precision 0.28 / recall 0.52). Diagnosis and fixes:

| Problem in v1 | Fix in v2 |
|---|---|
| Every overlapping window was decoded separately and the **union** of spans was submitted → many false positives (APP 244 FP, CASENO 350 FP on validation) | Token **probabilities are averaged** across windows (edge tokens down-weighted) and the document is decoded **once** |
| Greedy argmax BIO decoding, invalid tag sequences | **Constrained BIO Viterbi** + a tuned **O-bias** (precision/recall trade-off chosen on validation) |
| Class weights up to ~11× with O=0.2 → model over-predicts entities | Mild weights (inverse-frequency^0.25, clipped at 3, O=1) |
| New head trained at the same tiny LR (2e-5) as the backbone → under-trained head | Differential LR: backbone 6e-5, head 1e-3 |
| Causal Phase-1 decoder forced bidirectional with only a linear head | Variant sweep: causal vs bidirectional, linear vs **BiLSTM** head, optional **document-header context** |
| LORMA (Hadamard `W⊙(1+sBA)`) updated weights ~50× too little → F1 0.13 | Proper multiplicative low-rank update `W' = (I + s·BA)·W` |
| Other architectures: 1200 steps, frozen/partial, LR 1e-5..1e-4 → F1 0.05–0.22 | Same full fine-tuning recipe, same step budget |
| Padded-query rows in the SDPA mask were fully masked (NaN risk in fp16) | Only key padding is masked |
| Punctuation merged into BPE tokens broke boundaries (`...SC 1728].`) | Boundary rules learned from training gold |
| Only 7 validation docs; final model never saw them | 20 % validation split for selection, then **refit on all 70 docs** |
| Single model | **Probability-averaged ensemble** (greedy selection on validation), per-architecture submissions also written |

**v2.1 additions:** (1) a **span-boundary head** (BiLSTM + per-token start / end / type distributions, assembled by `BoundaryDecoder`) as a genuinely different modelling approach vs token-level BIO (`d_full_span`, set `RUN_SPAN=False` to skip, ~15 min); (2) cell 12b packages the **four official submissions** required by the spec (four different Phase-1 models, KD + LORMA + at least three techniques) plus two clearly labelled optional extras (span-head model, ensemble), each in its own folder under `artifacts/submissions/` with `prediction.json` + a ready-to-upload zip; (3) cell 11b adds the spec's error-analysis items (label confusions, boilerplate false positives, long-document effect).

Use a Kaggle GPU (T4). `QUICK=True` is only a smoke test. Outputs are saved under `/kaggle/working/artifacts/`. Total runtime for the full run is roughly 3–3.5 h on a T4 (set `RUN_EXTRA_PEFT=False` to save ~50 min).


## 0. Settings


In [ ]:
QUICK = False          # True = smoke test (~15 min). False = full training.
FORCE_RETRAIN = False  # True = ignore existing Phase-2 v2 checkpoints under /kaggle/working
REFIT = True           # retrain the top ensemble members on ALL labelled docs (train + val) at the end
REFIT_TOP_K = 3        # how many ensemble members to refit
MAX_LEN = 512          # tokens / window (Phase-1 max pos is 1024; 512 fits a T4)
STRIDE = 128           # eval/test window stride (probabilities are averaged over the overlap)
STRIDE_TRAIN = 256     # training window grid (start is randomly jittered every epoch)
HEAD_TOK = 96          # document-header tokens prepended to later windows in the *_ctx variant
VAL_FRACTION = 0.20
SEED = 42
RUN_SPAN = True        # span-boundary head experiment (start/end/type) in addition to token-level BIO

if QUICK:
    H = dict(max_steps=60, warmup=10, batch=2, eval_bs=4, log_every=10, eval_every=30,
             lora_r=8, lr_full=6e-5, lr_peft=5e-4, head_lr=1e-3, kd_steps=40)
else:
    H = dict(max_steps=2400, warmup=150, batch=4, eval_bs=8, log_every=100, eval_every=300,
             lora_r=16, lr_full=6e-5, lr_peft=5e-4, head_lr=1e-3, kd_steps=2000)
print("QUICK" if QUICK else "FULL", H)


In [ ]:
# transformers/tokenizers already on the Kaggle image (same stack as Phase 1).
# Internet On only if these fail.
import importlib
for m in ["torch", "tokenizers", "numpy", "pandas"]:
    mod = importlib.import_module(m)
    print(f"{m:14s} {getattr(mod, '__version__', 'ok')}")


## 1. Locate Phase-2 data + Phase-1 checkpoints

Kaggle unpacks uploads under `/kaggle/input/<dataset>/...` with unpredictable nesting. We search by filename.


In [ ]:
import os, json, random, math, time, shutil
from pathlib import Path

KAGGLE_INPUT = "/kaggle/input"
print("datasets:", os.listdir(KAGGLE_INPUT) if os.path.isdir(KAGGLE_INPUT) else "NO /kaggle/input")

def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

def find_any(base, names):
    for n in names:
        p = find_file(base, n)
        if p:
            return p
    return None

need_text = {
    "fold1_text": "fold1-text.json",
    "fold1_gold": "fold1-gold.json",
    "fold2_text": "fold2-text.json",
    "fold2_gold": "fold2-gold.json",
    "labels": "labels.txt",
    "fold3_text": "fold3-text.json",
}
PATHS = {}
for k, fn in need_text.items():
    p = find_file(KAGGLE_INPUT, fn)
    print(("OK   " if p else "MISS ") + fn + (f" -> {p}" if p else ""))
    PATHS[k] = p
assert all(PATHS[k] for k in need_text), "Upload the Phase_2_LNER zip (Train/ + Test/)."

PATHS["tokenizer"] = find_any(KAGGLE_INPUT, ["tokenizer.json"])
print("tokenizer:", PATHS["tokenizer"])
assert PATHS["tokenizer"], "Upload Phase-1 artifacts/tokenizer.json"

def find_phase1_checkpoint(arch):
    """Phase-1 layout: ckpt_tier1/<arch>_best.pt for all four architectures, ckpt_tier2/<WINNER>_best.pt for the
    final long run of the winner only.  Smoke / ablation / distillation checkpoints are ignored; Tier-2 beats Tier-1."""
    wanted = {f"{arch}_best.pt", f"{arch}_final.pt", f"{arch}.pt"}
    skip_dirs = ("ckpt_smoke", "ckpt_abl", "ckpt_distill")
    candidates = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        for name in files:
            if name in wanted:
                path = os.path.join(root, name)
                low_path = path.lower().replace("\\", "/")
                if any(d in low_path for d in skip_dirs):
                    continue
                score = 100 if name.endswith("_best.pt") else 0
                if "ckpt_tier2" in low_path:
                    score += 40
                elif "ckpt_tier1" in low_path:
                    score += 5
                candidates.append((score, path))
    return max(candidates, default=(0, None), key=lambda x: (x[0], x[1] or ""))[1]

CKPT = {}
for arch in ["decoder_only", "efficient", "chunk_recurrent", "enc_dec"]:
    CKPT[arch] = find_phase1_checkpoint(arch)
    print(f"ckpt {arch:18s} {CKPT[arch]}")
missing_arch = [a for a, p in CKPT.items() if p is None]
assert not missing_arch, ("Phase 2 requires experiments from all four Phase-1 architectures. "
                          f"Missing checkpoints: {missing_arch}. Upload decoder_only_best.pt, efficient_best.pt, "
                          "chunk_recurrent_best.pt, and enc_dec_best.pt (or matching *_final.pt files).")

os.makedirs("/kaggle/working/artifacts", exist_ok=True)
os.chdir("/kaggle/working")
print("cwd", os.getcwd())


## 2. Project files (Phase-1 backbones + L-NER v2 stack)


In [ ]:
%%writefile config.py
"""Minimal ModelConfig matching Phase-1 checkpoints."""
from dataclasses import dataclass

@dataclass
class ModelConfig:
    architecture: str = "decoder_only"
    vocab_size: int = 32000
    hidden_size: int = 768
    num_layers: int = 12
    num_heads: int = 12
    num_kv_heads: int = 4
    ffn_multiplier: float = 2.75
    max_position_embeddings: int = 1024
    dropout: float = 0.1
    local_window: int = 128
    global_every: int = 4
    chunk_size: int = 128
    memory_size: int = 32


In [ ]:
%%writefile models.py
"""
Four substantially different architectures for Phase-1 legal LM pretraining.
All kept in the 50M-300M parameter range by adjusting hidden_size / num_layers
in config.py.

    1. DecoderOnlyLM        - RoPE + GQA + SwiGLU pre-norm transformer (baseline)
    2. Seq2SeqLM            - Encoder-decoder transformer (natural fit for
                               "512-word prompt -> 128-word continuation")
    3. EfficientLocalGlobalLM - local sliding-window attention with periodic
                               full-attention layers (cheap long-range mixing)
    4. ChunkRecurrentLM     - novel: encodes the prompt in fixed-size chunks,
                               compresses each chunk into a memory vector via
                               a small recurrent update, decoder cross-attends
                               to accumulated memory + local window
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================================
# Shared primitives
# ============================================================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=2.75, dropout=0.1):
        super().__init__()
        hidden = int(dim * mult)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


def rotary_freqs(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cat([freqs, freqs], dim=-1)  # (seq_len, head_dim)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, freqs):
    cos, sin = freqs.cos(), freqs.sin()
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k


class GQAAttention(nn.Module):
    """Grouped-query causal self-attention with RoPE. Supports an optional
    additive attn_mask (for local-window / block patterns)."""

    def __init__(self, dim, num_heads, num_kv_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)
        self.drop = dropout

    def forward(self, x, freqs, attn_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, k, freqs)

        rep = self.num_heads // self.num_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)

        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=attn_mask is None,
            dropout_p=self.drop if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out)


def build_local_window_mask(T, window, device):
    """Boolean mask: True = attend. Causal + only last `window` tokens.
    (Boolean instead of an additive float mask: SDPA rejects a float32 mask
    when q/k/v are bf16/fp16 under autocast.)"""
    idx = torch.arange(T, device=device)
    causal = idx[None, :] <= idx[:, None]
    local = (idx[:, None] - idx[None, :]) < window
    return (causal & local)[None, None, :, :]


def build_causal_mask(T, device):
    idx = torch.arange(T, device=device)
    return (idx[None, :] <= idx[:, None])[None, None, :, :]


# ============================================================================
# 1. Decoder-only Transformer (baseline: RoPE + GQA + SwiGLU, pre-norm)
# ============================================================================
class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, attn_mask=None):
        x = x + self.attn(self.norm1(x), freqs, attn_mask)
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        for block in self.blocks:
            x = block(x, freqs)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, temperature=0.8, top_p=0.9,
                 top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=4,
                 eos_token_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            # BUG FIX: crop only the MODEL INPUT to the trained context. The old code overwrote `input_ids`
            # with its own crop, so the running sequence never grew past max_pos+1 tokens and generate.py
            # (which decodes out[0, prompt_len:]) got EMPTY text for every ctx-512 model and a truncated /
            # corrupted tail for long prompts on the ctx-1024 model.
            logits, _ = self.forward(input_ids[:, -self.cfg.max_position_embeddings:])
            next_logits = logits[:, -1, :].float() / max(temperature, 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk>
            if eos_token_id is None:
                next_logits[:, 2] = float("-inf")                # ...nor <eos> (would start an unrelated document)
            next_logits = _apply_repetition_penalty(next_logits, input_ids, repetition_penalty)
            next_logits = _block_repeat_ngrams(next_logits, input_ids, no_repeat_ngram_size)
            next_logits = _top_k_top_p_filter(next_logits, top_k, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if eos_token_id is not None and (next_token == eos_token_id).all():
                break
        return input_ids


def _apply_repetition_penalty(logits, generated, penalty):
    if penalty == 1.0:
        return logits
    for b in range(generated.size(0)):
        seen = torch.unique(generated[b])
        vals = logits[b, seen]
        logits[b, seen] = torch.where(vals < 0, vals * penalty, vals / penalty)
    return logits


def _block_repeat_ngrams(logits, generated, n):
    if n <= 0 or generated.size(1) < n:
        return logits
    B, T = generated.shape
    for b in range(B):
        seq = generated[b].tolist()
        if len(seq) < n - 1:
            continue
        prefix = tuple(seq[-(n - 1):])
        banned = set()
        for i in range(len(seq) - n + 1):
            if tuple(seq[i:i + n - 1]) == prefix:
                banned.add(seq[i + n - 1])
        for tok in banned:
            logits[b, tok] = float("-inf")
    return logits


def _top_k_top_p_filter(logits, top_k, top_p):
    if top_k > 0:
        top_k = min(top_k, logits.size(-1))
        kth_vals = torch.topk(logits, top_k, dim=-1).values[..., -1, None]
        logits = torch.where(logits < kth_vals, torch.full_like(logits, float("-inf")), logits)
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = F.softmax(sorted_logits, dim=-1)
        cum_probs = torch.cumsum(probs, dim=-1)
        remove = cum_probs > top_p
        remove[..., 1:] = remove[..., :-1].clone()
        remove[..., 0] = False
        sorted_logits[remove] = float("-inf")
        logits = torch.full_like(logits, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    return logits


# ============================================================================
# 2. Encoder-Decoder Transformer (natural fit: prompt -> continuation)
# ============================================================================
class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                           dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x))
        return x


class DecoderLayerCross(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                dropout=cfg.dropout, batch_first=True)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                 dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, memory, self_mask, mem_key_padding_mask=None):
        h = self.norm1(x)
        sa, _ = self.self_attn(h, h, h, attn_mask=self_mask, need_weights=False)
        x = x + sa
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory,
                                 key_padding_mask=mem_key_padding_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class Seq2SeqLM(nn.Module):
    """Encoder reads the 512-word prompt; decoder autoregressively produces
    the continuation. Uses learned positional embeddings (simplest to reason
    about across two separate sequences)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        half_layers = max(1, cfg.num_layers // 2)
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.pos_emb = nn.Embedding(cfg.max_position_embeddings, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(half_layers)])
        self.decoder = nn.ModuleList([DecoderLayerCross(cfg) for _ in range(half_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def _embed(self, ids):
        T = ids.size(1)
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        return self.drop(self.tok_emb(ids) + self.pos_emb(pos))

    def encode(self, src_ids, src_padding_mask=None):
        x = self._embed(src_ids)
        for layer in self.encoder:
            x = layer(x, key_padding_mask=src_padding_mask)
        return x

    def forward(self, src_ids, tgt_ids, labels=None, src_padding_mask=None):
        memory = self.encode(src_ids, src_padding_mask)
        x = self._embed(tgt_ids)
        T = tgt_ids.size(1)
        causal = torch.triu(torch.full((T, T), float("-inf"), device=tgt_ids.device), diagonal=1)
        for layer in self.decoder:
            x = layer(x, memory, causal, mem_key_padding_mask=src_padding_mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, eos_id, max_new_tokens, temperature=0.8,
                 top_p=0.9, top_k=50, repetition_penalty=1.2, no_repeat_ngram_size=4,
                 src_padding_mask=None):
        self.eval()
        memory = self.encode(src_ids, src_padding_mask)
        tgt = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            x = self._embed(tgt)
            T = tgt.size(1)
            causal = torch.triu(torch.full((T, T), float("-inf"), device=tgt.device), diagonal=1)
            h = x
            for layer in self.decoder:
                h = layer(h, memory, causal, mem_key_padding_mask=src_padding_mask)
            h = self.norm_f(h)
            next_logits = self.lm_head(h)[:, -1, :].float() / max(temperature, 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk> (<eos> ends the target)
            next_logits = _apply_repetition_penalty(next_logits, tgt, repetition_penalty)
            next_logits = _block_repeat_ngrams(next_logits, tgt, no_repeat_ngram_size)
            next_logits = _top_k_top_p_filter(next_logits, top_k, top_p)
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tgt = torch.cat([tgt, next_token], dim=1)
            if (next_token == eos_id).all():
                break
        return tgt


# ============================================================================
# 3. Efficient / lightweight Transformer: local sliding-window attention with
#    periodic full-attention layers for global mixing.
# ============================================================================
class EfficientLocalGlobalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        local_mask = build_local_window_mask(T, self.cfg.local_window, input_ids.device)
        full_mask = build_causal_mask(T, input_ids.device)
        for i, block in enumerate(self.blocks):
            is_global = (i + 1) % self.cfg.global_every == 0
            mask = full_mask if is_global else local_mask
            x = block(x, freqs, attn_mask=mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, **kw):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self.forward(input_ids[:, -self.cfg.max_position_embeddings:])   # crop the input only (bug fix)
            next_logits = logits[:, -1, :].float() / max(kw.get("temperature", 0.8), 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk>
            if kw.get("eos_token_id") is None:
                next_logits[:, 2] = float("-inf")                # ...nor <eos>
            next_logits = _apply_repetition_penalty(next_logits, input_ids, kw.get("repetition_penalty", 1.2))
            next_logits = _block_repeat_ngrams(next_logits, input_ids, kw.get("no_repeat_ngram_size", 4))
            next_logits = _top_k_top_p_filter(next_logits, kw.get("top_k", 50), kw.get("top_p", 0.9))
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if kw.get("eos_token_id") is not None and (next_token == kw["eos_token_id"]).all():
                break
        return input_ids


# ============================================================================
# 4. Chunk-Recurrent Hybrid (novel architecture):
#    Prompt is split into fixed-size chunks. Each chunk is summarized into a
#    memory vector via a GRU-style recurrent update (memory carried across
#    chunks, cheap O(T) alternative to full attention over very long
#    contexts). The decoder then does local causal attention over recent
#    tokens PLUS cross-attention to the accumulated memory bank -- letting it
#    reference facts from far earlier in a long legal prompt (e.g. an early
#    citation or defined term) without full quadratic attention.
# ============================================================================
class ChunkEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.chunk_size = cfg.chunk_size
        self.hidden = cfg.hidden_size
        self.intra_chunk_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                        dropout=cfg.dropout, batch_first=True)
        self.norm = RMSNorm(cfg.hidden_size)
        self.summarize = nn.Linear(cfg.hidden_size, cfg.hidden_size)
        self.recurrent_update = nn.GRUCell(cfg.hidden_size, cfg.hidden_size)

    def forward(self, x):
        """x: (B, T, H) prompt embeddings. Returns memory bank (B, num_chunks, H)."""
        B, T, H = x.shape
        n_chunks = math.ceil(T / self.chunk_size)
        pad = n_chunks * self.chunk_size - T
        if pad > 0:
            x = F.pad(x, (0, 0, 0, pad))
        chunks = x.view(B, n_chunks, self.chunk_size, H)

        state = torch.zeros(B, H, device=x.device, dtype=x.dtype)
        memories = []
        for c in range(n_chunks):
            seg = chunks[:, c]                      # (B, chunk_size, H)
            h = self.norm(seg)
            attn_out, _ = self.intra_chunk_attn(h, h, h, need_weights=False)
            seg = seg + attn_out
            pooled = self.summarize(seg.mean(dim=1))  # (B, H) mean-pool the chunk
            state = self.recurrent_update(pooled, state)
            memories.append(state)
        return torch.stack(memories, dim=1)  # (B, n_chunks, H)


class ChunkRecurrentBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.hidden_size)
        self.self_attn = GQAAttention(cfg.hidden_size, cfg.num_heads, cfg.num_kv_heads, cfg.dropout)
        self.norm2 = RMSNorm(cfg.hidden_size)
        self.cross_attn = nn.MultiheadAttention(cfg.hidden_size, cfg.num_heads,
                                                 dropout=cfg.dropout, batch_first=True)
        self.norm3 = RMSNorm(cfg.hidden_size)
        self.ffn = SwiGLU(cfg.hidden_size, cfg.ffn_multiplier, cfg.dropout)

    def forward(self, x, freqs, memory, mem_mask=None, attn_mask=None):
        x = x + self.self_attn(self.norm1(x), freqs, attn_mask)
        h = self.norm2(x)
        ca, _ = self.cross_attn(h, memory, memory, attn_mask=mem_mask, need_weights=False)
        x = x + ca
        x = x + self.ffn(self.norm3(x))
        return x


class ChunkRecurrentLM(nn.Module):
    """Decoder-only interface (matches the pure-continuation task) but with
    chunk-recurrent memory injected via cross-attention at every layer."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.hidden_size)
        self.drop = nn.Dropout(cfg.dropout)
        self.chunk_encoder = ChunkEncoder(cfg)
        self.blocks = nn.ModuleList([ChunkRecurrentBlock(cfg) for _ in range(cfg.num_layers)])
        self.norm_f = RMSNorm(cfg.hidden_size)
        self.lm_head = nn.Linear(cfg.hidden_size, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.head_dim = cfg.hidden_size // cfg.num_heads
        # learned "empty memory" slot so tokens in the first chunk have something to attend to
        self.null_mem = nn.Parameter(torch.zeros(1, 1, cfg.hidden_size))
        self.apply(self._init_weights)
        nn.init.normal_(self.null_mem, mean=0.0, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        freqs = rotary_freqs(T, self.head_dim, input_ids.device)
        memory = self.chunk_encoder(x.detach())            # (B, n_chunks, H): summary of chunk c
        n_chunks = memory.size(1)
        memory = torch.cat([self.null_mem.expand(B, -1, -1).to(memory.dtype), memory], dim=1)  # slot 0 = empty
        # CAUSALITY FIX: a token in chunk c may only read slot 0 and the summaries of chunks 0..c-1
        # (slots 1..c). The old code let every token read every chunk, including the chunk that
        # contains the very token it is asked to predict, so its val loss was artificially low.
        chunk_id = torch.arange(T, device=input_ids.device) // self.cfg.chunk_size
        slot = torch.arange(n_chunks + 1, device=input_ids.device)
        mem_mask = slot[None, :] > chunk_id[:, None]        # True = NOT allowed
        for block in self.blocks:
            x = block(x, freqs, memory, mem_mask)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens, **kw):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self.forward(input_ids[:, -self.cfg.max_position_embeddings:])   # crop the input only (bug fix)
            next_logits = logits[:, -1, :].float() / max(kw.get("temperature", 0.8), 1e-5)
            next_logits[:, [0, 1, 3]] = float("-inf")            # never sample <pad>/<bos>/<unk>
            if kw.get("eos_token_id") is None:
                next_logits[:, 2] = float("-inf")                # ...nor <eos>
            next_logits = _apply_repetition_penalty(next_logits, input_ids, kw.get("repetition_penalty", 1.2))
            next_logits = _block_repeat_ngrams(next_logits, input_ids, kw.get("no_repeat_ngram_size", 4))
            next_logits = _top_k_top_p_filter(next_logits, kw.get("top_k", 50), kw.get("top_p", 0.9))
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if kw.get("eos_token_id") is not None and (next_token == kw["eos_token_id"]).all():
                break
        return input_ids


# ============================================================================
# Factory + param counting
# ============================================================================
def build_model(cfg):
    arch = cfg.architecture
    if arch == "decoder_only":
        return DecoderOnlyLM(cfg)
    if arch == "enc_dec":
        return Seq2SeqLM(cfg)
    if arch == "efficient":
        return EfficientLocalGlobalLM(cfg)
    if arch == "chunk_recurrent":
        return ChunkRecurrentLM(cfg)
    raise ValueError(f"Unknown architecture: {arch}")


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================================
# Checkpoint / AMP helpers shared by train / eval / generate / distill
# ============================================================================
def get_amp_dtype():
    """bf16 on Ampere+ GPUs, fp16 on T4/P100 (no native bf16), None on CPU."""
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def load_checkpoint(path, device):
    """Load a checkpoint written by train.py / distill.py. Configs are stored as plain dicts,
    so this works on torch>=2.6 where torch.load defaults to weights_only=True and cannot
    unpickle a dataclass (the old torch.load(path) calls would have failed)."""
    from config import ModelConfig
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    if isinstance(cfg, dict):
        cfg = ModelConfig(**cfg)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model, cfg, ckpt


In [ ]:
%%writefile lner.py
"""
Phase-2 L-NER v2: BIO token classification on Phase-1 Legal LMs.

What changed vs v1 (all aimed at strict span Macro-F1):
  * Overlapping windows are merged by AVERAGING token probabilities (edge-down-weighted), then the
    document is decoded ONCE.  v1 decoded every window and took the UNION of spans, which is the main
    reason precision was ~0.28 (every window edge / noisy window added false positives).
  * Constrained BIO Viterbi decoding (no I-X after O / other type) + a tunable "O bias" that trades
    precision against recall on the validation documents.
  * Mild class weights (v1 weights of up to ~11x with O=0.2 made the model over-predict entities).
  * Differential learning rates: the new head learns ~20x faster than the pretrained backbone.
  * Optional BiLSTM head (gives a causal Phase-1 decoder access to right context) and a switch for
    causal vs bidirectional attention at fine-tune time.
  * Optional document-header context (first N tokens are prepended to later windows, labels ignored),
    so party names in the caption can inform APP / RESP mentions deep in the judgment.
  * Random-jitter training windows (augmentation) instead of a fixed 4x-redundant window grid.
  * Boundary refinement rules learned from the training gold (trailing/leading punctuation).
  * Fixed LORMA (multiplicative low-rank adaptation that actually trains), fixed NaN-prone padding mask.
  * Probability-level ensembling across models (all share one tokenizer).
"""
from __future__ import annotations

import json
import math
import os
import random
import time
from bisect import bisect_right
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from config import ModelConfig
from models import build_model, rotary_freqs, count_params, get_amp_dtype

LNER_VERSION = "v2.1"


# ---------------------------------------------------------------------------
# IO helpers
# ---------------------------------------------------------------------------
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def parse_entity(item):
    """Return (start, end, label, text_or_None) from list or dict gold items."""
    if isinstance(item, dict):
        s = item.get("start", item.get("start_offset", item.get("begin")))
        e = item.get("end", item.get("end_offset", item.get("finish")))
        lab = item.get("label", item.get("entity", item.get("type")))
        txt = item.get("text", item.get("entity_text", item.get("span")))
        return int(s), int(e), str(lab), txt
    if isinstance(item, (list, tuple)):
        s, e, lab = int(item[0]), int(item[1]), str(item[2])
        txt = item[3] if len(item) > 3 else None
        return s, e, lab, txt
    raise TypeError(f"bad entity item: {type(item)}")


def detect_end_exclusive(texts: dict, golds: dict, n_check=400) -> bool:
    exc_ok = inc_ok = checked = 0
    for did, ents in golds.items():
        text = texts.get(str(did), texts.get(did))
        if text is None:
            continue
        for item in ents:
            s, e, lab, txt = parse_entity(item)
            if not txt:
                continue
            checked += 1
            if text[s:e] == txt:
                exc_ok += 1
            if e <= len(text) and text[s:e + 1] == txt:
                inc_ok += 1
            if checked >= n_check:
                break
        if checked >= n_check:
            break
    exclusive = exc_ok >= inc_ok
    print(f"offset convention: exclusive-end matches={exc_ok}  inclusive-end matches={inc_ok}  -> "
          f"{'EXCLUSIVE [s:e)' if exclusive else 'INCLUSIVE [s:e]'}")
    return exclusive


def normalize_gold(texts, golds, exclusive: bool, labels_allowed: set):
    out, dropped = {}, 0
    for did, ents in golds.items():
        key = str(did)
        text = texts.get(key, texts.get(did, ""))
        norm = []
        for item in ents:
            s, e, lab, txt = parse_entity(item)
            if not exclusive:
                e = e + 1
            s = max(0, s)
            e = min(len(text), e)
            if e <= s or lab not in labels_allowed:
                dropped += 1
                continue
            norm.append([s, e, lab])
        out[key] = norm
    print(f"normalized gold entities: {sum(len(v) for v in out.values())}  dropped={dropped}")
    return out


# ---------------------------------------------------------------------------
# BIO scheme, tokenization, alignment
# ---------------------------------------------------------------------------
def build_label_maps(type_labels: List[str]):
    id2lab = ["O"]
    for t in type_labels:
        id2lab.append(f"B-{t}")
        id2lab.append(f"I-{t}")
    return {l: i for i, l in enumerate(id2lab)}, id2lab


def encode_doc(tokenizer, text):
    enc = tokenizer.encode(text)
    ids = list(enc.ids)
    offs = [(int(a), int(b)) for a, b in enc.offsets]
    special_ids = set()
    for t in ["<pad>", "<bos>", "<eos>", "<unk>"]:
        tid = tokenizer.token_to_id(t)
        if tid is not None:
            special_ids.add(tid)
    special = [(ids[i] in special_ids and offs[i] == (0, 0)) for i in range(len(ids))]
    return ids, offs, special


def align_bio(offsets, gold_spans, lab2id, special_mask):
    """Gold [s,e) spans -> BIO ids. First token overlapping an entity is B-, the rest I-.
    Special / empty-offset tokens get -100."""
    n = len(offsets)
    labels = [-100] * n
    valid = [i for i in range(n) if (not special_mask[i]) and offsets[i][1] > offsets[i][0]]
    o_id = lab2id["O"]
    for i in valid:
        labels[i] = o_id
    if not valid:
        return labels
    run_end = np.maximum.accumulate(np.array([offsets[i][1] for i in valid]))
    run_end = run_end.tolist()
    for s, e, lab in sorted(gold_spans):
        k = bisect_right(run_end, s)  # first valid token whose running-max end > s
        first = True
        while k < len(valid):
            i = valid[k]
            cs, ce = offsets[i]
            if cs >= e:
                break
            if ce <= s:
                k += 1
                continue
            if labels[i] == o_id:
                labels[i] = lab2id[("B-" if first else "I-") + lab]
                first = False
            k += 1
    return labels


def trim_span_ws(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def apply_rule(text, s, e, rule):
    st, en = rule if rule else ("", "")
    while s < e and (text[s].isspace() or text[s] in st):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in en):
        e -= 1
    return s, e


def decode_tags(offsets, tags, id2lab, text, rules=None):
    """Hard BIO tags -> exact half-open char spans. B always starts a new entity."""
    spans = []
    cur = None

    def close():
        nonlocal cur
        if cur is not None:
            s, e, typ = cur
            s, e = apply_rule(text, max(0, s), min(len(text), e), (rules or {}).get(typ))
            if e > s:
                spans.append([s, e, typ])
        cur = None

    for i, (cs, ce) in enumerate(offsets):
        t = tags[i]
        if ce <= cs or t < 0:
            close()
            continue
        lab = id2lab[t]
        if lab == "O":
            close()
        elif lab.startswith("B-") or cur is None or cur[2] != lab[2:]:
            close()
            cur = [cs, ce, lab[2:]]
        else:
            cur[1] = ce
    close()
    return spans


# ---------------------------------------------------------------------------
# Documents and windows
# ---------------------------------------------------------------------------
@dataclass
class Doc:
    did: str
    text: str
    ids: List[int]
    offs: List[Tuple[int, int]]
    special: List[bool]
    labels: List[int]
    n: int


def build_docs(texts, golds, tokenizer, lab2id, labeled=True):
    docs = {}
    n_tok = n_ent = 0
    for did, text in texts.items():
        did = str(did)
        ids, offs, special = encode_doc(tokenizer, text)
        if labeled:
            labels = align_bio(offs, golds.get(did, []), lab2id, special)
        else:
            labels = [(-100 if special[i] else lab2id["O"]) for i in range(len(ids))]
        n_tok += sum(l != -100 for l in labels)
        n_ent += sum(l > 0 for l in labels)
        docs[did] = Doc(did, text, ids, offs, special, labels, len(ids))
    print(f"docs={len(docs)} tokens={sum(d.n for d in docs.values())} labeled_tokens={n_tok} "
          f"entity_tokens={n_ent} entity_frac={n_ent / max(1, n_tok):.3f}")
    return docs


@dataclass
class Win:
    did: str
    ids: List[int]
    labels: List[int]
    ctx: int      # number of leading context-only tokens (header prefix)
    start: int    # doc token index of the first non-context token
    m: int        # number of non-context tokens


def make_window(doc: Doc, start: int, max_len: int, head_tok: int) -> Win:
    n = doc.n
    if head_tok and start >= head_tok:
        body = max_len - head_tok
        end = min(n, start + body)
        ids = doc.ids[:head_tok] + doc.ids[start:end]
        labels = [-100] * head_tok + doc.labels[start:end]
        ctx = head_tok
    else:
        end = min(n, start + max_len)
        ids = doc.ids[start:end]
        labels = doc.labels[start:end]
        ctx = 0
    return Win(doc.did, ids, labels, ctx, start, end - start)


def eval_windows(docs: Dict[str, Doc], max_len: int, stride: int, head_tok: int) -> List[Win]:
    out = []
    for doc in docs.values():
        start = 0
        while True:
            w = make_window(doc, start, max_len, head_tok)
            out.append(w)
            if w.start + w.m >= doc.n:
                break
            start += stride
    return out


class TrainWindows(Dataset):
    """One item per ~stride tokens; the window start is re-jittered on every access."""

    def __init__(self, docs: Dict[str, Doc], max_len: int, head_tok: int, stride: int):
        self.docs = list(docs.values())
        self.max_len, self.head_tok, self.stride = max_len, head_tok, stride
        self.items = []
        for di, d in enumerate(self.docs):
            for s in range(0, max(1, d.n), stride):
                self.items.append((di, s))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        di, s = self.items[i]
        d = self.docs[di]
        if s > 0:
            s = s + random.randint(-self.stride // 2, self.stride // 2)
        s = max(0, min(s, max(0, d.n - self.max_len // 2)))
        return make_window(d, s, self.max_len, self.head_tok)


def collate_wins(batch: List[Win], pad_id: int):
    T = max(len(w.ids) for w in batch)
    B = len(batch)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    attn = torch.zeros((B, T), dtype=torch.long)
    lab = torch.full((B, T), -100, dtype=torch.long)
    for i, w in enumerate(batch):
        L = len(w.ids)
        ids[i, :L] = torch.tensor(w.ids, dtype=torch.long)
        attn[i, :L] = 1
        lab[i, :L] = torch.tensor(w.labels, dtype=torch.long)
    return {"input_ids": ids, "attention_mask": attn, "labels": lab, "wins": batch}


# ---------------------------------------------------------------------------
# Backbone hidden states
# ---------------------------------------------------------------------------
def sdpa_mask(attention_mask, bidirectional=True, local_window=None, global_full=False):
    """Boolean SDPA mask (True = attend). Only KEY padding is masked; masking padded queries too would
    create all-False rows -> NaN under SDPA, which then poisons later layers."""
    B, T = attention_mask.shape
    device = attention_mask.device
    mask = attention_mask[:, None, None, :].bool()
    if not bidirectional:
        mask = mask & torch.tril(torch.ones((T, T), dtype=torch.bool, device=device))
    if local_window is not None and not global_full:
        idx = torch.arange(T, device=device)
        local = (idx[:, None] - idx[None, :]).abs() < local_window
        if not bidirectional:
            local = local & (idx[None, :] <= idx[:, None])
        mask = mask & local
    return mask


def encode_hidden(backbone, architecture, input_ids, attention_mask, bidirectional=True):
    B, T = input_ids.shape
    if architecture == "enc_dec":
        return backbone.encode(input_ids, src_padding_mask=(attention_mask == 0))

    x = backbone.drop(backbone.tok_emb(input_ids))
    freqs = rotary_freqs(T, backbone.head_dim, input_ids.device)

    if architecture == "decoder_only":
        m = sdpa_mask(attention_mask, bidirectional=bidirectional)
        for block in backbone.blocks:
            x = block(x, freqs, m)
        return backbone.norm_f(x)

    if architecture == "efficient":
        for i, block in enumerate(backbone.blocks):
            is_global = (i + 1) % backbone.cfg.global_every == 0
            m = sdpa_mask(attention_mask, bidirectional=bidirectional,
                          local_window=None if is_global else backbone.cfg.local_window,
                          global_full=is_global)
            x = block(x, freqs, m)
        return backbone.norm_f(x)

    if architecture == "chunk_recurrent":
        memory = backbone.chunk_encoder(x)
        memory = torch.cat([backbone.null_mem.expand(B, -1, -1).to(memory.dtype), memory], dim=1)
        m = sdpa_mask(attention_mask, bidirectional=True)
        for block in backbone.blocks:
            x = block(x, freqs, memory, None, m)
        return backbone.norm_f(x)

    raise ValueError(architecture)


def span_targets(labels):
    """BIO ids (B,L) -> start / end targets (B,L) in {0 = none, 1..T = entity type}, -100 = ignore.
    Id layout: 1 = B-type1, 2 = I-type1, 3 = B-type2, ...  A token is a START if it carries a B- tag and an END
    if it is the last token of an entity (next label is not an I- of the same type)."""
    lab = labels
    pos = lab > 0
    typ = torch.where(pos, (lab + 1) // 2, torch.zeros_like(lab))
    is_b = pos & (lab % 2 == 1)
    nxt = torch.cat([lab[:, 1:], torch.full_like(lab[:, :1], -100)], dim=1)
    nxt_same_i = (nxt > 0) & (nxt % 2 == 0) & ((nxt + 1) // 2 == typ)
    is_end = pos & ~nxt_same_i
    end_unknown = pos & (nxt == -100)      # next label ignored (window edge): the end is not known
    start_t = torch.where(is_b, typ, torch.zeros_like(typ)).masked_fill(lab == -100, -100)
    end_t = torch.where(is_end, typ, torch.zeros_like(typ)).masked_fill((lab == -100) | end_unknown, -100)
    return start_t, end_t


class NERModel(nn.Module):
    """head = 'linear' (BIO), 'bilstm' (BiLSTM + BIO) or 'span' (BiLSTM + start/end/type boundary head).
    The span head predicts, per token, a start distribution and an end distribution over {none, type_1..type_T}
    (output = concat of the two softmaxes); spans are assembled by BoundaryDecoder instead of BIO decoding."""

    def __init__(self, backbone, hidden_size, num_labels, architecture, dropout=0.1,
                 head="linear", bidir=True, lstm_hidden=256):
        super().__init__()
        self.backbone = backbone
        self.architecture = architecture
        self.head_kind = head
        self.bidir = bool(bidir)
        self.num_types = (num_labels - 1) // 2
        self.drop = nn.Dropout(dropout)
        if head in ("bilstm", "span"):
            self.lstm = nn.LSTM(hidden_size, lstm_hidden, batch_first=True, bidirectional=True)
            feat = 2 * lstm_hidden
        else:
            feat = hidden_size
        self.out_dim = 2 * (self.num_types + 1) if head == "span" else num_labels
        self.classifier = nn.Linear(feat, self.out_dim)

    def features(self, input_ids, attention_mask):
        return encode_hidden(self.backbone, self.architecture, input_ids, attention_mask,
                             bidirectional=self.bidir)

    def classify(self, h, attention_mask):
        h = self.drop(h)
        if self.head_kind in ("bilstm", "span"):
            T = h.size(1)
            lengths = attention_mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(h, lengths, batch_first=True, enforce_sorted=False)
            out, _ = self.lstm(packed)
            h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=T)
            h = self.drop(h)
        return self.classifier(h)

    def span_loss(self, logits, labels, class_weights=None):
        K = self.num_types + 1
        st, en = span_targets(labels)
        w = None
        if class_weights is not None:      # [O, B-type1, B-type2, ...] taken from the BIO weights
            w = torch.cat([class_weights[:1], class_weights[1::2]])
        lf = logits.float()
        ls = F.cross_entropy(lf[..., :K].reshape(-1, K), st.reshape(-1), ignore_index=-100, weight=w)
        le = F.cross_entropy(lf[..., K:].reshape(-1, K), en.reshape(-1), ignore_index=-100, weight=w)
        return ls + le

    def to_probs(self, logits):
        lf = logits.float()
        if self.head_kind == "span":
            K = self.num_types + 1
            return torch.cat([torch.softmax(lf[..., :K], dim=-1), torch.softmax(lf[..., K:], dim=-1)], dim=-1)
        return torch.softmax(lf, dim=-1)

    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        logits = self.classify(self.features(input_ids, attention_mask), attention_mask)
        loss = None
        if labels is not None:
            if self.head_kind == "span":
                loss = self.span_loss(logits, labels, class_weights)
            else:
                loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), labels.reshape(-1),
                                       ignore_index=-100, weight=class_weights)
        return logits, loss


# ---------------------------------------------------------------------------
# PEFT: LoRA (additive) and LORMA (multiplicative low-rank)
# ---------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + s * B A x"""

    def __init__(self, base: nn.Linear, r=16, alpha=32):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        out, inn = base.weight.shape
        self.A = nn.Parameter(torch.empty(r, inn))
        self.B = nn.Parameter(torch.zeros(out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = F.linear(x, self.base.weight, self.base.bias)
        return h + self.scale * F.linear(F.linear(x, self.A), self.B)


class LoRMALinear(nn.Module):
    """Low-Rank Multiplicative Adaptation: W' = (I + s * B A) W, i.e. the frozen weight is multiplied by
    a low-rank matrix instead of being added to.  Computed without materialising W':
        y = h + s * B (A h),   h = W x.
    (v1 used a Hadamard product W * (1 + s*BA); since |W| ~ 0.02 that update was ~50x too small to
    learn anything, which is why LORMA scored 0.13.)"""

    def __init__(self, base: nn.Linear, r=16, alpha=32):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        out, _ = base.weight.shape
        self.A = nn.Parameter(torch.empty(r, out))
        self.B = nn.Parameter(torch.zeros(out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = F.linear(x, self.base.weight, self.base.bias)
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)


TARGET_LINEAR = {"q_proj", "k_proj", "v_proj", "o_proj", "w1", "w2", "w3", "summarize"}


def inject_peft(module, kind="lora", r=16, alpha=32, names=TARGET_LINEAR):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in names:
            wrap = LoRALinear if kind == "lora" else LoRMALinear
            setattr(module, name, wrap(child, r=r, alpha=alpha))
        else:
            inject_peft(child, kind, r, alpha, names)


def freeze_backbone(backbone, mode="full"):
    """mode: full | last4 | bitfit | peft-ready (everything frozen; adapters are added afterwards)."""
    for p in backbone.parameters():
        p.requires_grad = False
    if mode == "full":
        for n, p in backbone.named_parameters():
            p.requires_grad = "lm_head" not in n
    elif mode == "last4":
        n_blocks = len(backbone.blocks) if hasattr(backbone, "blocks") else 0
        for n, p in backbone.named_parameters():
            if "lm_head" in n:
                continue
            if n_blocks and any(f"blocks.{i}." in n for i in range(max(0, n_blocks - 4), n_blocks)):
                p.requires_grad = True
            elif "norm_f" in n or "tok_emb" in n:
                p.requires_grad = True
            elif hasattr(backbone, "encoder") and ("encoder" in n):
                p.requires_grad = True
    elif mode == "bitfit":
        for n, p in backbone.named_parameters():
            if n.endswith("bias") or "norm" in n.lower():
                p.requires_grad = True


def trainable_params(model):
    n_all = sum(p.numel() for p in model.parameters())
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return n_all, n_tr


def read_cfg(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]
    if isinstance(cfg, dict):
        cfg = ModelConfig(**{k: v for k, v in cfg.items() if k in ModelConfig.__dataclass_fields__})
    return cfg


def load_phase1(path, device="cpu"):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    if isinstance(cfg, dict):
        cfg = ModelConfig(**{k: v for k, v in cfg.items() if k in ModelConfig.__dataclass_fields__})
    model = build_model(cfg)
    model.load_state_dict(ckpt["model"], strict=True)
    print(f"loaded {os.path.basename(path)}  arch={cfg.architecture}  (all tensors matched)")
    return model, cfg, ckpt


# ---------------------------------------------------------------------------
# Class weights, optimizer, training loop
# ---------------------------------------------------------------------------
def class_weights_from_docs(docs: Dict[str, Doc], num_labels, power=0.25, clip=3.0, device="cpu"):
    cnt = Counter()
    for d in docs.values():
        cnt.update(l for l in d.labels if l != -100)
    c = np.array([max(cnt.get(i, 0), 1) for i in range(num_labels)], dtype=np.float64)
    w = np.clip((c[0] / c) ** power, 1.0, clip)
    w[0] = 1.0
    return torch.tensor(w, dtype=torch.float32, device=device), cnt


def make_optimizer(model, lr, head_lr, wd=0.01):
    buckets = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        head = n.startswith("classifier") or n.startswith("lstm")
        no_decay = p.ndim == 1 or n.endswith(".bias") or n.endswith(".A") or n.endswith(".B")
        buckets.setdefault((head, no_decay), []).append(p)
    groups = []
    for (head, no_decay), params in buckets.items():
        base = head_lr if head else lr
        groups.append({"params": params, "weight_decay": 0.0 if no_decay else wd,
                       "lr": base, "base_lr": base})
    return torch.optim.AdamW(groups, betas=(0.9, 0.98))


def _train_loop(model, loader, loss_fn, eval_fn, device, lr, head_lr, max_steps, warmup,
                log_every, amp_dtype, eval_every, tag="", grad_clip=1.0, min_eval_step=0):
    model.to(device)
    opt = make_optimizer(model, lr, head_lr)
    use_amp = device == "cuda" and amp_dtype is not None
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)
    params = [p for p in model.parameters() if p.requires_grad]
    best_f1, best_state, best_step, step, t0 = -1.0, None, 0, 0, time.time()
    history = []
    model.train()
    while step < max_steps:
        for batch in loader:
            step += 1
            if step < warmup:
                factor = step / max(1, warmup)
            else:
                prog = (step - warmup) / max(1, max_steps - warmup)
                factor = 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * factor
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda" if device == "cuda" else "cpu",
                                dtype=amp_dtype or torch.float16, enabled=use_amp):
                loss, info = loss_fn(batch)
            if not torch.isfinite(loss):
                print(f"  [{tag}] step {step}: non-finite loss, skipping batch", flush=True)
                opt.zero_grad(set_to_none=True)
            else:
                if scaler.is_enabled():
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(params, grad_clip)
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(params, grad_clip)
                    opt.step()
            if step % log_every == 0:
                extra = " ".join(f"{k} {v:.4f}" for k, v in (info or {}).items())
                print(f"  [{tag}] step {step:5d}  loss {loss.item():.4f} {extra}  "
                      f"lr {opt.param_groups[0]['lr']:.2e}  {(time.time() - t0) / 60:.1f} min", flush=True)
            if eval_fn is not None and ((step % eval_every == 0 and step >= min_eval_step) or step == max_steps):
                f1 = eval_fn(model)
                history.append((step, f1))
                print(f"  [{tag}] EVAL step {step}  val macro-F1={f1:.4f}", flush=True)
                if f1 > best_f1:
                    best_f1, best_step = f1, step
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                model.train()
            if step >= max_steps:
                break
    if eval_fn is not None and best_state is not None:
        model.load_state_dict(best_state)
    print(f"[{tag}] done  best_val_F1={best_f1:.4f}@{best_step}  steps={step}  "
          f"{(time.time() - t0) / 60:.1f} min", flush=True)
    return best_f1, best_step, history


def train_ner(model, loader, eval_fn, device, lr, head_lr, max_steps, warmup, class_weights,
              log_every, amp_dtype, eval_every, tag="", min_eval_step=0):
    def loss_fn(b):
        ids = b["input_ids"].to(device)
        attn = b["attention_mask"].to(device)
        y = b["labels"].to(device)
        _, loss = model(ids, attn, labels=y, class_weights=class_weights)
        return loss, None

    return _train_loop(model, loader, loss_fn, eval_fn, device, lr, head_lr, max_steps, warmup,
                       log_every, amp_dtype, eval_every, tag, min_eval_step=min_eval_step)


def distill_train(teacher, student, loader, eval_fn, device, lr, head_lr, max_steps, warmup,
                  temperature, alpha, beta_hidden, class_weights, log_every, amp_dtype, eval_every,
                  tag="kd", min_eval_step=0):
    """Student loss = alpha*T^2*KL(student||teacher) + (1-alpha)*CE(gold) + beta*(1-cos(hidden_s, hidden_t))."""
    teacher.to(device).eval()
    for p in teacher.parameters():
        p.requires_grad = False
    n_lab = student.classifier.out_features

    def loss_fn(b):
        ids = b["input_ids"].to(device)
        attn = b["attention_mask"].to(device)
        y = b["labels"].to(device)
        with torch.no_grad():
            th = teacher.features(ids, attn)
            tl = teacher.classify(th, attn)
        sh = student.features(ids, attn)
        sl = student.classify(sh, attn)
        mask = y.ne(-100)
        ce = F.cross_entropy(sl.float().reshape(-1, n_lab), y.reshape(-1), ignore_index=-100,
                             weight=class_weights)
        kd = F.kl_div(F.log_softmax(sl[mask].float() / temperature, dim=-1),
                      F.softmax(tl[mask].float() / temperature, dim=-1),
                      reduction="batchmean") * (temperature ** 2)
        cos = 1.0 - F.cosine_similarity(sh[mask].float(), th[mask].float(), dim=-1).mean()
        loss = alpha * kd + (1 - alpha) * ce + beta_hidden * cos
        return loss, {"ce": ce.item(), "kd": kd.item(), "cos": cos.item()}

    return _train_loop(student, loader, loss_fn, eval_fn, device, lr, head_lr, max_steps, warmup,
                       log_every, amp_dtype, eval_every, tag, min_eval_step=min_eval_step)


# ---------------------------------------------------------------------------
# Inference: per-token probabilities merged over overlapping windows
# ---------------------------------------------------------------------------
@torch.no_grad()
def predict_probs(model, wins: List[Win], docs: Dict[str, Doc], pad_id, device, amp_dtype,
                  batch_size, num_labels):
    model.eval()
    model.to(device)
    num_labels = getattr(model, "out_dim", num_labels)   # span head has a different output width
    acc = {did: np.zeros((d.n, num_labels), dtype=np.float32) for did, d in docs.items()}
    wsum = {did: np.zeros(d.n, dtype=np.float32) for did, d in docs.items()}
    use_amp = device == "cuda" and amp_dtype is not None
    order = sorted(range(len(wins)), key=lambda i: len(wins[i].ids))  # similar lengths -> less padding
    for b0 in range(0, len(order), batch_size):
        batch = [wins[i] for i in order[b0:b0 + batch_size]]
        col = collate_wins(batch, pad_id)
        ids = col["input_ids"].to(device)
        attn = col["attention_mask"].to(device)
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu",
                            dtype=amp_dtype or torch.float16, enabled=use_amp):
            logits, _ = model(ids, attn)
        pr = (model.to_probs(logits) if hasattr(model, "to_probs")
              else torch.softmax(logits.float(), dim=-1)).cpu().numpy()
        for i, w in enumerate(batch):
            n = docs[w.did].n
            m = w.m
            j = np.arange(m)
            left = (j + 1) if w.start > 0 else np.full(m, 10 ** 6)
            right = (m - j) if (w.start + m) < n else np.full(m, 10 ** 6)
            wt = np.minimum(np.minimum(left, right), 128).astype(np.float32)
            acc[w.did][w.start:w.start + m] += pr[i, w.ctx:w.ctx + m] * wt[:, None]
            wsum[w.did][w.start:w.start + m] += wt
    out = {}
    for did in acc:
        out[did] = acc[did] / np.maximum(wsum[did], 1e-6)[:, None]
    return out


def avg_probs(list_of_probs, weights=None):
    weights = weights or [1.0] * len(list_of_probs)
    tot = float(sum(weights))
    out = {}
    for did in list_of_probs[0]:
        out[did] = sum(w * p[did] for w, p in zip(weights, list_of_probs)) / tot
    return out


# ---------------------------------------------------------------------------
# Constrained Viterbi decoding + span decoder
# ---------------------------------------------------------------------------
def build_transitions(id2lab):
    L = len(id2lab)
    trans = np.zeros((L, L), dtype=np.float64)
    for i, a in enumerate(id2lab):
        for j, b in enumerate(id2lab):
            if b.startswith("I-") and not (a != "O" and a[2:] == b[2:]):
                trans[i, j] = -1e9
    start = np.array([-1e9 if l.startswith("I-") else 0.0 for l in id2lab])
    return trans, start


def viterbi(logp, trans, start):
    n, L = logp.shape
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    score = start + logp[0]
    bp = np.zeros((n, L), dtype=np.int16)
    cols = np.arange(L)
    for t in range(1, n):
        cand = score[:, None] + trans
        idx = cand.argmax(0)
        bp[t] = idx
        score = cand[idx, cols] + logp[t]
    tags = np.zeros(n, dtype=np.int64)
    tags[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        tags[t - 1] = bp[t, tags[t]]
    return tags


class SpanDecoder:
    def __init__(self, id2lab, rules=None, o_bias=0.0, use_viterbi=True):
        self.id2lab = id2lab
        self.rules = rules or {}
        self.o_bias = o_bias
        self.use_viterbi = use_viterbi
        self.trans, self.start = build_transitions(id2lab)

    def decode(self, probs, doc: Doc):
        lp = np.log(np.clip(probs.astype(np.float64), 1e-8, 1.0))
        lp[:, 0] += self.o_bias
        if self.use_viterbi:
            tags = viterbi(lp, self.trans, self.start)
        else:
            tags = lp.argmax(1)
        return decode_tags(doc.offs, tags.tolist(), self.id2lab, doc.text, self.rules)

    def decode_all(self, probs_by_doc, docs):
        return {did: self.decode(probs_by_doc[did], docs[did]) for did in docs}


class BoundaryDecoder:
    """Span-extraction style decoding for the span head. probs[:, :K] = start distribution over
    {none, type_1..type_T}, probs[:, K:] = end distribution.  The O-bias is added to the 'none' log-prob of both
    distributions (same precision/recall knob as in BIO decoding).  Each predicted start of type t is closed at the
    nearest following token whose end-argmax is t (before the next predicted start, at most max_tok tokens);
    if none exists, the most probable end inside that window is used when its probability >= min_end_p."""

    def __init__(self, id2lab, rules=None, o_bias=0.0, max_tok=256, min_end_p=0.10):
        self.id2lab = id2lab
        self.types = [l[2:] for l in id2lab if l.startswith("B-")]
        self.K = len(self.types) + 1
        self.rules = rules or {}
        self.o_bias = o_bias
        self.max_tok = max_tok
        self.min_end_p = min_end_p

    def decode(self, probs, doc: Doc):
        K = self.K
        n = probs.shape[0]
        lp = np.log(np.clip(probs.astype(np.float64), 1e-8, 1.0))
        ls, le = lp[:, :K].copy(), lp[:, K:].copy()
        ls[:, 0] += self.o_bias
        le[:, 0] += self.o_bias
        st = ls.argmax(1)
        en = le.argmax(1)
        valid = np.array([ce > cs for cs, ce in doc.offs], dtype=bool)
        st = np.where(valid, st, 0)
        starts = np.flatnonzero(st > 0)
        spans = []
        for k, i in enumerate(starts):
            i = int(i)
            t = int(st[i])
            nxt = int(starts[k + 1]) if k + 1 < len(starts) else n
            hi = min(nxt, i + self.max_tok)
            cand = np.flatnonzero((en[i:hi] == t) & valid[i:hi])
            if len(cand):
                j = i + int(cand[0])
            else:
                seg = probs[i:hi, K + t] * valid[i:hi]
                jj = int(seg.argmax())
                if seg[jj] < self.min_end_p:
                    continue
                j = i + jj
            typ = self.types[t - 1]
            s_, e_ = apply_rule(doc.text, max(0, doc.offs[i][0]), min(len(doc.text), doc.offs[j][1]),
                                self.rules.get(typ))
            if e_ > s_:
                spans.append([s_, e_, typ])
        return spans

    def decode_all(self, probs_by_doc, docs):
        return {did: self.decode(probs_by_doc[did], docs[did]) for did in docs}


def make_decoder(id2lab, rules, o_bias, probs_by_doc, use_viterbi=True):
    """BIO probabilities (width = #BIO labels) -> SpanDecoder; span-head probabilities -> BoundaryDecoder."""
    width = next(iter(probs_by_doc.values())).shape[1]
    if width != len(id2lab):
        return BoundaryDecoder(id2lab, rules, o_bias)
    return SpanDecoder(id2lab, rules, o_bias, use_viterbi)


# ---------------------------------------------------------------------------
# Metrics + boundary-rule learning
# ---------------------------------------------------------------------------
def strict_macro_f1(gold: dict, pred: dict, type_labels: List[str]):
    per = {t: Counter() for t in type_labels}
    for did in gold:
        g = gold[did]
        p = pred.get(did, pred.get(str(did), []))
        gs = {t: set() for t in type_labels}
        ps = {t: set() for t in type_labels}
        for s, e, lab in g:
            if lab in gs:
                gs[lab].add((s, e))
        for s, e, lab in p:
            if lab in ps:
                ps[lab].add((s, e))
        for t in type_labels:
            tp = len(gs[t] & ps[t])
            per[t]["tp"] += tp
            per[t]["fp"] += len(ps[t]) - tp
            per[t]["fn"] += len(gs[t]) - tp
    rows, f1s = [], []
    for t in type_labels:
        tp, fp, fn = per[t]["tp"], per[t]["fp"], per[t]["fn"]
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        f1s.append(f1)
        rows.append({"label": t, "tp": tp, "fp": fp, "fn": fn, "P": prec, "R": rec, "F1": f1})
    return {"macro_f1": float(np.mean(f1s)), "macro_p": float(np.mean([r["P"] for r in rows])),
            "macro_r": float(np.mean([r["R"] for r in rows])), "per_class": rows}


STRIP_CANDS = list(".,;:[](){}\"'-/&")


def learn_strip_rules(docs: Dict[str, Doc], gold: dict, raw_spans: dict, type_labels, min_gain=2):
    """Greedy per-label learning of leading/trailing punctuation to strip, so that tokenizer-merged
    punctuation ("... SC 1728]." etc.) does not break strict boundaries. Fit on round-tripped gold only."""
    rules = {}
    for t in type_labels:
        items = [(did, s, e) for did, sp in raw_spans.items() for s, e, l in sp if l == t]
        gold_t = {(did, s, e) for did, g in gold.items() for s, e, l in g if l == t}

        def tp(rule):
            n = 0
            for did, s, e in items:
                s2, e2 = apply_rule(docs[did].text, s, e, rule)
                if (did, s2, e2) in gold_t:
                    n += 1
            return n

        cur, best = ("", ""), tp(("", ""))
        for side in (1, 0):  # end first, then start
            for c in STRIP_CANDS:
                if c in cur[side]:
                    continue
                cand = (cur[0] + c, cur[1]) if side == 0 else (cur[0], cur[1] + c)
                v = tp(cand)
                if v >= best + min_gain:
                    cur, best = cand, v
        rules[t] = cur
    return rules


## 3. Load corpus, verify character offsets, tokenize, learn boundary rules

The notebook detects exclusive vs inclusive gold ends, builds BIO labels with the Phase-1 tokenizer, measures the exact gold → tokens → spans round-trip, and learns per-label punctuation-trimming rules from the *training gold only* so tokenizer-merged punctuation does not break strict boundaries.


In [ ]:
import numpy as np
import pandas as pd
import torch
from collections import Counter
from copy import deepcopy
from tokenizers import Tokenizer
from IPython.display import display

from config import ModelConfig
from models import build_model, count_params, get_amp_dtype

# Make sure we import the just-written lner.py, not a stale module/bytecode copy.
import importlib, sys
for cached_pyc in Path("__pycache__").glob("lner.*.pyc"):
    cached_pyc.unlink()
sys.modules.pop("lner", None)
import lner as _lner
assert getattr(_lner, "LNER_VERSION", "") == "v2.1", "Stale lner.py: restart the Kaggle session and run from the top."

from lner import (
    load_json, detect_end_exclusive, normalize_gold, build_label_maps, build_docs, eval_windows,
    TrainWindows, collate_wins, class_weights_from_docs, load_phase1, read_cfg, freeze_backbone,
    inject_peft, trainable_params, NERModel, train_ner, distill_train, predict_probs, avg_probs,
    SpanDecoder, BoundaryDecoder, make_decoder, strict_macro_f1, learn_strip_rules, decode_tags,
)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
amp = get_amp_dtype()
print("device", device, "amp", amp)

tok = Tokenizer.from_file(PATHS["tokenizer"])
tok.post_processor = None
pad_id = tok.token_to_id("<pad>") or 0
print("vocab", tok.get_vocab_size(), "pad_id", pad_id)

type_labels = [ln.strip() for ln in open(PATHS["labels"], encoding="utf-8") if ln.strip()]
type_labels = [t for t in type_labels if t not in ("O", "PAD") and not t.startswith("#")]
EXPECTED = ["APP","RESP","A.COUNSEL","R.COUNSEL","JUDGE","WIT","AUTH","COURT","STAT","PREC","DATE","CASENO"]
if set(type_labels) != set(EXPECTED):
    print("WARNING labels.txt", type_labels, " - using file order, expected", EXPECTED)
lab2id, id2lab = build_label_maps(type_labels)
print("BIO labels", len(id2lab))

def load_fold(p):
    return {str(k): v for k, v in load_json(p).items()}

texts, golds_raw = {}, {}
texts.update(load_fold(PATHS["fold1_text"])); texts.update(load_fold(PATHS["fold2_text"]))
golds_raw.update(load_fold(PATHS["fold1_gold"])); golds_raw.update(load_fold(PATHS["fold2_gold"]))
test_texts = load_fold(PATHS["fold3_text"])
print("train docs", len(texts), "gold keys", len(golds_raw), "test docs", len(test_texts))

exclusive = detect_end_exclusive(texts, golds_raw)
golds = normalize_gold(texts, golds_raw, exclusive, set(type_labels))

lab_c = Counter(lab for ents in golds.values() for _, _, lab in ents)
print(pd.Series(lab_c).sort_values(ascending=False).to_string())

docs_all = build_docs(texts, golds, tok, lab2id, labeled=True)
docs_test = build_docs(test_texts, {}, tok, lab2id, labeled=False)
lens = [len(v) for v in texts.values()]
print(f"char len  min={min(lens)} median={int(np.median(lens))} max={max(lens)}")


In [ ]:
# Round-trip: gold -> BIO -> decoded spans (before and after the learned boundary rules).
rt_raw = {did: decode_tags(d.offs, d.labels, id2lab, d.text, None) for did, d in docs_all.items()}
rt0 = strict_macro_f1(golds, rt_raw, type_labels)
print("ROUND-TRIP (no rules)    macro-F1", round(rt0["macro_f1"], 4))

RULES = learn_strip_rules(docs_all, golds, rt_raw, type_labels)
print("learned boundary rules (start-strip, end-strip):", {k: v for k, v in RULES.items() if v != ("", "")})
rt_rules = {did: decode_tags(d.offs, d.labels, id2lab, d.text, RULES) for did, d in docs_all.items()}
rt = strict_macro_f1(golds, rt_rules, type_labels)
print("ROUND-TRIP (with rules)  macro-F1", round(rt["macro_f1"], 4), "P", round(rt["macro_p"], 4), "R", round(rt["macro_r"], 4))
display(pd.DataFrame(rt["per_class"]))

ROUND_TRIP_MIN_F1 = 0.97
if rt["macro_f1"] < ROUND_TRIP_MIN_F1:
    raise RuntimeError(f"Span alignment too inaccurate ({rt['macro_f1']:.4f}); stop before training.")

shown = 0
for did in rt_rules:
    g = {tuple(x) for x in golds[did]}; p = {tuple(x) for x in rt_rules[did]}
    for s, e, lab in sorted(g - p):
        cand = [q for q in p - g if q[2] == lab and q[0] == s]
        if cand and shown < 5:
            print(f"doc={did} {lab} gold={docs_all[did].text[s:e]!r} decoded={docs_all[did].text[cand[0][0]:cand[0][1]]!r}")
            shown += 1


In [ ]:
# Held-out validation split by document (no official dev set).
doc_ids = sorted(docs_all.keys())
random.Random(SEED).shuffle(doc_ids)
n_val = max(2, int(len(doc_ids) * VAL_FRACTION))
val_ids = set(doc_ids[:n_val])
train_ids = [d for d in doc_ids if d not in val_ids]
docs_train = {d: docs_all[d] for d in train_ids}
docs_val = {d: docs_all[d] for d in val_ids}
val_gold = {d: golds[d] for d in val_ids}
print("train docs", len(docs_train), "val docs", len(docs_val))

cw_preview, cnt_preview = class_weights_from_docs(docs_train, len(id2lab))
print("class weights (mild):", {id2lab[i]: round(cw_preview[i].item(), 2) for i in range(len(id2lab))})

_WIN_CACHE = {}
def get_wins(kind, head_tok):
    key = (kind, head_tok)
    if key not in _WIN_CACHE:
        docs = {"val": docs_val, "test": docs_test}[kind]
        _WIN_CACHE[key] = eval_windows(docs, MAX_LEN, STRIDE, head_tok)
        print(f"windows[{kind}, head_tok={head_tok}] = {len(_WIN_CACHE[key])}")
    return _WIN_CACHE[key]

def probs_for(model, kind, head_tok):
    docs = {"val": docs_val, "test": docs_test}[kind]
    return predict_probs(model, get_wins(kind, head_tok), docs, pad_id, device, amp, H["eval_bs"], len(id2lab))

def score(probs, o_bias=0.0):
    pred = make_decoder(id2lab, RULES, o_bias, probs).decode_all(probs, docs_val)
    return strict_macro_f1(val_gold, pred, type_labels), pred

def make_loader(docs, head_tok):
    ds = TrainWindows(docs, MAX_LEN, head_tok, STRIDE_TRAIN)
    return torch.utils.data.DataLoader(ds, batch_size=H["batch"], shuffle=True, num_workers=0,
                                       collate_fn=lambda b: collate_wins(b, pad_id))


## 4. Experiment factory

`fresh_model(spec)` loads a Phase-1 checkpoint, freezes/adapts it (`full`, `lora`, `lorma`) and attaches the BIO head (`linear` or `bilstm`). `run_experiment` trains with differential LRs, keeps the best validation checkpoint, then stores the **window-averaged token probabilities** for validation and test so everything downstream (decoding, O-bias, ensembling) is cheap.


In [ ]:
import json as _json

RESULTS = "artifacts/p2v2_results.jsonl"
os.makedirs("artifacts/ckpt_lner", exist_ok=True)
if FORCE_RETRAIN and os.path.exists(RESULTS):
    os.remove(RESULTS)

SPEC_OF, VAL_P, TEST_P, VAL_SCORE = {}, {}, {}, {}

def log_result(row):
    with open(RESULTS, "a", encoding="utf-8") as f:
        f.write(_json.dumps(row) + "\n")
    print(row)

def show_results():
    cols = ["tag", "arch", "method", "head", "bidir", "head_tok", "val_macro_f1", "val_P", "val_R",
            "params_M", "trainable_M", "train_min", "val_infer_s", "peak_gpu_GB"]
    rows = [r for r in (_json.loads(l) for l in open(RESULTS))] if os.path.exists(RESULTS) else []
    if not rows:
        print("no results yet"); return None
    df = pd.DataFrame(rows).drop_duplicates("tag", keep="last")
    df = df[[c for c in cols if c in df.columns]].sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
    display(df)
    return df

def fresh_model(spec):
    backbone, cfg, _ = load_phase1(CKPT[spec["arch"]], "cpu")
    m = spec["method"]
    if m in ("lora", "lorma"):
        r = spec.get("r") or H["lora_r"]
        freeze_backbone(backbone, "peft-ready")
        inject_peft(backbone, kind=m, r=r, alpha=2 * r)
    else:
        freeze_backbone(backbone, m)
    return NERModel(backbone, cfg.hidden_size, len(id2lab), cfg.architecture, 0.1, spec["head"], spec["bidir"])

def skeleton_kd(spec):
    cfg = deepcopy(read_cfg(CKPT["decoder_only"]))
    cfg.num_layers = spec["student_depth"]
    return NERModel(build_model(cfg), cfg.hidden_size, len(id2lab), "decoder_only", 0.1, spec["head"], spec["bidir"])

def restore(blob):
    spec = blob["spec"]
    model = skeleton_kd(spec) if spec["method"] == "kd" else fresh_model(spec)
    model.load_state_dict(blob["model"], strict=True)
    return model

def build_student(teacher, spec):
    """Half-depth student initialised from evenly spaced layers of the *fine-tuned* teacher (+ its head)."""
    tb = teacher.backbone
    tcfg = tb.cfg
    depth = spec["student_depth"]
    scfg = deepcopy(tcfg); scfg.num_layers = depth
    sb = build_model(scfg)
    ts, ss = tb.state_dict(), sb.state_dict()
    layers = np.linspace(0, tcfg.num_layers - 1, depth).round().astype(int).tolist()
    transfer = {}
    for k, v in ss.items():
        if k.startswith("blocks."):
            _, lid, suf = k.split(".", 2)
            sk = f"blocks.{layers[int(lid)]}.{suf}"
        else:
            sk = k
        if sk in ts and ts[sk].shape == v.shape:
            transfer[k] = ts[sk].detach().cpu().clone()
    sb.load_state_dict(transfer, strict=False)
    st = NERModel(sb, scfg.hidden_size, len(id2lab), "decoder_only", 0.1, teacher.head_kind, teacher.bidir)
    st.classifier.load_state_dict({k: v.detach().cpu() for k, v in teacher.classifier.state_dict().items()})
    if teacher.head_kind == "bilstm":
        st.lstm.load_state_dict({k: v.detach().cpu() for k, v in teacher.lstm.state_dict().items()})
    for p in sb.parameters():
        p.requires_grad = True
    print(f"KD student: {len(transfer)}/{len(ss)} tensors copied from teacher layers {layers}; depth {tcfg.num_layers}->{depth}")
    return st

def run_experiment(tag, spec, steps, lr, teacher=None):
    spec = dict(spec); spec.update(tag=tag, steps=steps, lr=lr)
    path = f"artifacts/ckpt_lner/{tag}.pt"
    ht = spec["head_tok"]
    train_min, best_step, peak = 0.0, None, 0.0
    if os.path.exists(path) and not FORCE_RETRAIN:
        blob = torch.load(path, map_location="cpu", weights_only=False)
        model = restore(blob)
        train_min, best_step = blob.get("train_min", 0.0), blob.get("best_step")
        print(f"[reuse] {path}")
    else:
        model = build_student(teacher, spec) if spec["method"] == "kd" else fresh_model(spec)
        n_all, n_tr = trainable_params(model)
        print(f"=== {tag}: {spec['arch']} / {spec['method']} / head={spec['head']} / bidir={spec['bidir']} / "
              f"head_tok={ht}: {n_all/1e6:.2f}M total, {n_tr/1e6:.2f}M trainable ({100*n_tr/max(n_all,1):.2f}%)")
        loader = make_loader(docs_train, ht)
        cw, _ = class_weights_from_docs(docs_train, len(id2lab), device=device)
        eval_fn = lambda m: score(probs_for(m, "val", ht))[0]["macro_f1"]
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        if spec["method"] == "kd":
            _, best_step, _ = distill_train(teacher, model, loader, eval_fn, device, lr, H["head_lr"], steps,
                                            H["warmup"], 2.0, 0.7, 0.5, cw, H["log_every"], amp,
                                            H["eval_every"], tag=tag)
        else:
            _, best_step, _ = train_ner(model, loader, eval_fn, device, lr, H["head_lr"], steps, H["warmup"],
                                        cw, H["log_every"], amp, H["eval_every"], tag=tag)
        train_min = (time.time() - t0) / 60
        peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
        torch.save({"model": model.state_dict(), "spec": spec, "train_min": train_min, "best_step": best_step}, path)
    model.to(device).eval()
    t0 = time.time()
    vp = probs_for(model, "val", ht)
    val_infer_s = time.time() - t0
    tp = probs_for(model, "test", ht)
    met, _ = score(vp)
    n_all, n_tr = trainable_params(model)
    VAL_P[tag], TEST_P[tag], SPEC_OF[tag], VAL_SCORE[tag] = vp, tp, spec, met["macro_f1"]
    log_result({"tag": tag, "arch": spec["arch"], "method": spec["method"], "head": spec["head"],
                "bidir": spec["bidir"], "head_tok": ht, "val_macro_f1": round(met["macro_f1"], 4),
                "val_P": round(met["macro_p"], 4), "val_R": round(met["macro_r"], 4),
                "params_M": round(n_all / 1e6, 2), "trainable_M": round(n_tr / 1e6, 3),
                "train_min": round(train_min, 1), "best_step": best_step,
                "val_infer_s": round(val_infer_s, 1), "peak_gpu_GB": round(peak, 2)})
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return met["macro_f1"]


## 5. Exp 1 — Full fine-tuning of Phase-1 `decoder_only`: head / attention-mode / context sweep

Four *methodologically different* modelling choices (all full fine-tuning): linear vs BiLSTM head, causal (as pretrained) vs bidirectional attention, and document-header context. The best one becomes the default recipe for every other experiment and the KD teacher.


In [ ]:
VARIANTS = {
    "d_full_bidir_lin":      dict(head="linear", bidir=True,  head_tok=0),
    "d_full_causal_lstm":    dict(head="bilstm", bidir=False, head_tok=0),
    "d_full_bidir_lstm":     dict(head="bilstm", bidir=True,  head_tok=0),
    "d_full_bidir_lstm_ctx": dict(head="bilstm", bidir=True,  head_tok=HEAD_TOK),
}
for tag, v in VARIANTS.items():
    run_experiment(tag, dict(arch="decoder_only", method="full", r=None, **v), H["max_steps"], H["lr_full"])

best_variant = max(VARIANTS, key=lambda t: VAL_SCORE[t])
BASE = {k: SPEC_OF[best_variant][k] for k in ("head", "bidir", "head_tok")}
print("BEST VARIANT", best_variant, VAL_SCORE[best_variant], BASE)
_ = show_results()


## 5b. Exp 1b — Span-boundary head (span-based vs token-based modelling)

Instead of per-token BIO tags, the head predicts for every token a **start** distribution and an **end** distribution over {none, type₁..type₁₂}. `BoundaryDecoder` turns them into spans (each start is closed at the nearest token whose end-argmax has the same type). Same full fine-tuning recipe, backbone and context settings as the best BIO variant above, so the comparison isolates the head / decoding paradigm. Its token probabilities have a different width, so it is evaluated and submitted on its own (not averaged into the BIO ensemble).


In [ ]:
if RUN_SPAN:
    run_experiment("d_full_span",
                   dict(arch="decoder_only", method="full", r=None, head="span",
                        bidir=BASE["bidir"], head_tok=BASE["head_tok"]),
                   H["max_steps"], H["lr_full"])
_ = show_results()


## 6. Exp 2 / 3 — LoRA and **LORMA** (compulsory) on `decoder_only`

Frozen Phase-1 weights + rank-r adapters on Q/K/V/O and SwiGLU projections; the head is trained at a higher LR. LoRA is additive (`W + sBA`); LORMA is multiplicative (`(I + sBA)·W`).


In [ ]:
def base_spec(arch, method, r=None):
    return dict(arch=arch, method=method, r=r, head=BASE["head"],
                bidir=(BASE["bidir"] if arch in ("decoder_only", "efficient") else True),
                head_tok=BASE["head_tok"])

run_experiment("d_lora",  base_spec("decoder_only", "lora",  H["lora_r"]), H["max_steps"], H["lr_peft"])
run_experiment("d_lorma", base_spec("decoder_only", "lorma", H["lora_r"]), H["max_steps"], H["lr_peft"])
_ = show_results()


## 7. Exp 4 — Knowledge distillation (mandatory)

Teacher: the best full fine-tuned `decoder_only` model above. Student: half depth (12 → 6 layers), initialised from evenly spaced *fine-tuned teacher* layers and the teacher head; loss = `0.7·T²·KL(student‖teacher)` + `0.3·CE(gold)` + `0.5·(1 − cos(hidden_student, hidden_teacher))`, T = 2.


In [ ]:
tblob = torch.load(f"artifacts/ckpt_lner/{best_variant}.pt", map_location="cpu", weights_only=False)
teacher = restore(tblob).to(device).eval()
kd_spec = dict(arch="decoder_only", method="kd", r=None, head=BASE["head"], bidir=BASE["bidir"],
               head_tok=BASE["head_tok"], student_depth=6)
run_experiment("d_kd_student", kd_spec, H["kd_steps"], H["lr_full"] * 2, teacher=teacher)
del teacher
if device == "cuda":
    torch.cuda.empty_cache()
_ = show_results()


## 8. Other Phase-1 architectures (all four are used)

Full fine-tuning, LoRA and LORMA on `efficient` (local/global attention), `chunk_recurrent` and `enc_dec` (encoder half, bidirectional), with the same recipe and step budget as `decoder_only`. v1 gave these only 1200 steps with frozen/partial training, which is why they scored 0.05–0.22. (For `enc_dec` the adapters attach to the SwiGLU projections only, because `nn.MultiheadAttention` has no separate q/k/v Linear modules.)


In [ ]:
OTHER = [("e", "efficient"), ("c", "chunk_recurrent"), ("ed", "enc_dec")]
RUN_EXTRA_PEFT = True    # LoRA + LORMA on every other Phase-1 architecture (set False to save ~50 min)
for pre, arch in OTHER:
    if not CKPT.get(arch):
        print(f"[skip] no Phase-1 checkpoint for {arch}"); continue
    run_experiment(f"{pre}_full", base_spec(arch, "full"), H["max_steps"], H["lr_full"])
    if RUN_EXTRA_PEFT:
        run_experiment(f"{pre}_lora",  base_spec(arch, "lora",  H["lora_r"]), H["max_steps"], H["lr_peft"])
        run_experiment(f"{pre}_lorma", base_spec(arch, "lorma", H["lora_r"]), H["max_steps"], H["lr_peft"])
results_df = show_results()


## 9. Ensemble selection + decoding calibration (validation)

All models share the Phase-1 tokenizer, so per-token probabilities can be averaged. Greedy forward selection adds a model only if validation strict macro-F1 improves by > 0.002; then the **O-bias** (precision/recall knob for Viterbi decoding) is tuned.


In [ ]:
cands = [t for t in VAL_P if VAL_SCORE[t] > 0.05 and SPEC_OF[t]["head"] != "span"]   # span probs have another width
ens = [max(cands, key=lambda t: VAL_SCORE[t])]
best_s = VAL_SCORE[ens[0]]
while len(ens) < 6:
    trial = None
    for t in cands:
        if t in ens:
            continue
        s = score(avg_probs([VAL_P[x] for x in ens + [t]]))[0]["macro_f1"]
        if s > best_s + 0.002 and (trial is None or s > trial[1]):
            trial = (t, s)
    if trial is None:
        break
    ens.append(trial[0]); best_s = trial[1]
    print("ensemble +", trial[0], "->", round(best_s, 4))
print("ENSEMBLE", ens, "val macro-F1", round(best_s, 4))

ens_val = avg_probs([VAL_P[t] for t in ens])
rows, BEST_O_BIAS, best_f = [], 0.0, -1
for b in [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0]:
    m, _ = score(ens_val, b)
    rows.append({"o_bias": b, "F1": round(m["macro_f1"], 4), "P": round(m["macro_p"], 4), "R": round(m["macro_r"], 4)})
    if m["macro_f1"] > best_f + 1e-9:
        best_f, BEST_O_BIAS = m["macro_f1"], b
display(pd.DataFrame(rows))
print("BEST O_BIAS", BEST_O_BIAS, "val macro-F1", round(best_f, 4))


## 10. Refit top members on all 70 labelled documents

Validation was only used for model selection; the final models also learn from the validation documents (same recipe and step budget, no early stopping). Test probabilities of the original (train-only) and refit models are averaged.


In [ ]:
REFIT_TAGS = [t for t in ens if SPEC_OF[t]["method"] != "kd"][:REFIT_TOP_K] if REFIT else []
print("refitting:", REFIT_TAGS)
for t in REFIT_TAGS:
    rtag = t + "_refit"
    spec = dict(SPEC_OF[t]); spec["tag"] = rtag
    path = f"artifacts/ckpt_lner/{rtag}.pt"
    ht = spec["head_tok"]
    if os.path.exists(path) and not FORCE_RETRAIN:
        model = restore(torch.load(path, map_location="cpu", weights_only=False)); print("[reuse]", path)
    else:
        model = fresh_model(spec)
        loader = make_loader(docs_all, ht)
        cw, _ = class_weights_from_docs(docs_all, len(id2lab), device=device)
        lr = H["lr_peft"] if spec["method"] in ("lora", "lorma") else H["lr_full"]
        train_ner(model, loader, None, device, lr, H["head_lr"], spec["steps"], H["warmup"], cw,
                  H["log_every"], amp, H["eval_every"], tag=rtag)
        torch.save({"model": model.state_dict(), "spec": spec}, path)
    TEST_P[rtag] = probs_for(model.to(device), "test", ht)
    SPEC_OF[rtag] = spec
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
print("test prob sets:", sorted(TEST_P))


## 10b. Decoding-strategy comparison (validation)

Compares span-decoding strategies on the best single model and the ensemble: greedy argmax BIO vs constrained Viterbi (with and without the tuned O-bias).


In [ ]:
rows = []
bt = max((t for t in VAL_SCORE if SPEC_OF[t]["head"] != "span"), key=VAL_SCORE.get)
for name, probs in [(f"best single ({bt})", VAL_P[bt]), ("ensemble", ens_val)]:
    for dec_name, use_vit, ob in [("argmax", False, 0.0), ("viterbi", True, 0.0), (f"viterbi + O-bias {BEST_O_BIAS}", True, BEST_O_BIAS)]:
        pred = SpanDecoder(id2lab, RULES, ob, use_viterbi=use_vit).decode_all(probs, docs_val)
        m = strict_macro_f1(val_gold, pred, type_labels)
        rows.append({"model": name, "decoding": dec_name, "F1": round(m["macro_f1"], 4), "P": round(m["macro_p"], 4), "R": round(m["macro_r"], 4)})
if "d_full_span" in VAL_P:
    for ob in [-1.0, 0.0, 0.5, 1.0, 2.0]:
        m, _ = score(VAL_P["d_full_span"], ob)
        rows.append({"model": "span head (d_full_span)", "decoding": f"boundary decoder, O-bias {ob}",
                     "F1": round(m["macro_f1"], 4), "P": round(m["macro_p"], 4), "R": round(m["macro_r"], 4)})
display(pd.DataFrame(rows))


## 11. Error analysis on the validation ensemble

Per-type F1, boundary-only errors (right label, wrong offsets), pure false positives / negatives.


In [ ]:
met_ens, val_pred = score(ens_val, BEST_O_BIAS)
print("ensemble val strict macro-F1", round(met_ens["macro_f1"], 4), "P", round(met_ens["macro_p"], 4), "R", round(met_ens["macro_r"], 4))
display(pd.DataFrame(met_ens["per_class"]).sort_values("F1"))

shift, type_fp, type_fn, confuse = Counter(), Counter(), Counter(), Counter()
for did, gents in val_gold.items():
    g = {(s, e, l) for s, e, l in gents}
    p = {(s, e, l) for s, e, l in val_pred.get(did, [])}
    for s, e, l in g - p:
        if any(pp[2] == l and not (pp[1] <= s or pp[0] >= e) for pp in p):
            shift[l] += 1
        elif any(pp[2] != l and pp[0] == s and pp[1] == e for pp in p):
            confuse[l] += 1
        else:
            type_fn[l] += 1
    for s, e, l in p - g:
        if not any(gg[2] == l and not (gg[1] <= s or gg[0] >= e) for gg in g):
            type_fp[l] += 1
print("boundary-shift (overlap, same label):", dict(shift))
print("label confusion (same span, other label):", dict(confuse))
print("pure FN:", dict(type_fn))
print("pure FP:", dict(type_fp))


## 11b. Spec focus areas: label confusion, boilerplate false positives, long documents


In [ ]:
from collections import defaultdict
pair, fp_txt = Counter(), defaultdict(Counter)
for did, gents in val_gold.items():
    g = {(s, e, l) for s, e, l in gents}
    p = {(s, e, l) for s, e, l in val_pred.get(did, [])}
    gspan = {(s, e): l for s, e, l in g}
    for s, e, l in p - g:
        if (s, e) in gspan:
            pair[(gspan[(s, e)], l)] += 1          # same span, wrong label: gold -> predicted
        elif not any(not (gg[1] <= s or gg[0] >= e) for gg in g):
            fp_txt[l][docs_val[did].text[s:e].strip()[:60]] += 1   # pure false positive (boilerplate candidates)
print("label confusions (gold -> predicted, identical span):", pair.most_common(10))
print("most frequent pure false-positive strings per label:")
for l in type_labels:
    top = fp_txt[l].most_common(3)
    if top:
        print(f"  {l:10s}", top)

rows = []
for did, d in docs_val.items():
    g = {tuple(x) for x in val_gold[did]}
    p = {tuple(x) for x in val_pred.get(did, [])}
    tp = len(g & p)
    rows.append({"doc": did, "tokens": d.n, "windows": len(range(0, max(1, d.n - MAX_LEN) + STRIDE, STRIDE)),
                 "gold": len(g), "pred": len(p), "micro_F1": round(2 * tp / max(1, len(g) + len(p)), 4)})
print("per-document micro-F1 vs length (long documents need several overlapping windows):")
display(pd.DataFrame(rows).sort_values("tokens"))


## 12. Test prediction (`fold3-text.json`) + submission files

**Spec-compliant set:** `strict1..4_<arch>_<technique>.zip` = four submissions on four *different* Phase-1 models that include LORMA and KD and at least three different techniques (KD on decoder_only; the best LORMA among the other architectures; the remaining architectures use their best technique not yet used). `tech_<technique>_<arch>.zip` = best single model per technique. `submission_lner.zip` = probability-averaged ensemble of several models: an *optional extra* for the leaderboard, not one of the four required experiments. The 4th field (`entity_text`) is ignored by the scorer.


In [ ]:
import zipfile

def members_for(tags):
    ps = []
    for t in tags:
        if t in TEST_P: ps.append(TEST_P[t])
        if (t + "_refit") in TEST_P: ps.append(TEST_P[t + "_refit"])
    return ps

def write_submission(tags, name, o_bias):
    probs = avg_probs(members_for(tags))
    pred = make_decoder(id2lab, RULES, o_bias, probs).decode_all(probs, docs_test)
    out = {}
    for did, d in docs_test.items():
        out[str(did)] = [[s, e, lab, d.text[s:e]] for s, e, lab in pred.get(did, []) if 0 <= s < e <= len(d.text)]
    assert set(out) == set(test_texts), "prediction keys != test keys"
    pj = f"artifacts/{name}.json"
    with open(pj, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    zp = f"artifacts/{name}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    print(f"wrote {zp}  members={len(members_for(tags))}  entities={sum(len(v) for v in out.values())}")
    return out

out = write_submission(ens, "prediction", BEST_O_BIAS)
shutil.copy("artifacts/prediction.zip", "artifacts/submission_lner.zip")
shutil.copy("artifacts/prediction.zip", "artifacts/submission.zip")

# ---- STRICT FOUR (spec 8.2): four submissions, four DIFFERENT Phase-1 models, >=3 techniques, LORMA + KD included
def tech_of(t):
    return "kd" if SPEC_OF[t]["method"] == "kd" else SPEC_OF[t]["method"]
def best_tag(pred):
    ts = [t for t in VAL_SCORE if pred(t) and SPEC_OF[t]["head"] != "span"]   # span model is added separately in 12b
    return max(ts, key=lambda t: VAL_SCORE[t]) if ts else None
def best_bias(tag):
    bm = {b: score(VAL_P[tag], b)[0]["macro_f1"] for b in [-0.5, 0.0, 0.5, 1.0, 1.5]}
    return max(bm, key=bm.get)

STRICT = {}
STRICT["decoder_only"] = best_tag(lambda t: tech_of(t) == "kd")                                   # KD (student of decoder_only)
lorma_other = best_tag(lambda t: tech_of(t) == "lorma" and SPEC_OF[t]["arch"] != "decoder_only")   # compulsory LORMA
if lorma_other:
    STRICT[SPEC_OF[lorma_other]["arch"]] = lorma_other
used = {tech_of(t) for t in STRICT.values()}
for arch in ["decoder_only", "efficient", "chunk_recurrent", "enc_dec"]:
    if arch in STRICT:
        continue
    t = best_tag(lambda t: SPEC_OF[t]["arch"] == arch and tech_of(t) not in used and tech_of(t) != "kd")         or best_tag(lambda t: SPEC_OF[t]["arch"] == arch and tech_of(t) != "kd")
    if t:
        STRICT[arch] = t; used.add(tech_of(t))
print("STRICT FOUR:", {a: (t, tech_of(t), round(VAL_SCORE[t], 4)) for a, t in STRICT.items()})
for i, (arch, t) in enumerate(STRICT.items(), 1):
    write_submission([t], f"strict{i}_{arch}_{tech_of(t)}", best_bias(t))

# ---- best single model per technique (full / lora / lorma / kd), any architecture
for tech in ["full", "lora", "lorma", "kd"]:
    t = best_tag(lambda t: tech_of(t) == tech)
    if t:
        write_submission([t], f"tech_{tech}_{SPEC_OF[t]['arch']}", best_bias(t))
print("Spec-compliant submissions: artifacts/strict1..4_*.zip.  Optional extras: artifacts/tech_*.zip and the ensemble artifacts/submission_lner.zip")


## 12b. Official four submissions (+ optional extras)

The spec requires **four experiments/submissions on four different Phase-1 models**, with at least three different adaptation techniques, **LORMA** and **knowledge distillation** included. `STRICT` (cell 12) already selects them from the validation results; this cell packages them:

* `official1..4_<architecture>_<technique>/`: the four spec-compliant submissions
* `extra_span_head_decoder_only/`: span-boundary head model (extra experiment for the span-vs-token analysis)
* `extra_ensemble/`: probability-averaged ensemble (extra)

Each folder holds `prediction.json` and an upload-ready zip (zip root = `prediction.json`). Everything is also bundled in `artifacts/all_submissions.zip`. Upload the four `official*` zips; the extras are optional.


In [ ]:
import hashlib

SUB_DIR = "artifacts/submissions"
shutil.rmtree(SUB_DIR, ignore_errors=True)
os.makedirs(SUB_DIR)

# spec check (8.2): four different Phase-1 models, >=3 techniques, LORMA and KD both present
assert len(STRICT) == 4, f"need four different Phase-1 models, got {list(STRICT)}"
techs = {tech_of(t) for t in STRICT.values()}
assert {"kd", "lorma"} <= techs and len(techs) >= 3, f"technique mix does not satisfy the spec: {techs}"

plan = []   # (folder, tags, o_bias)
for n, (arch, t) in enumerate(STRICT.items(), 1):
    plan.append((f"official{n}_{arch}_{tech_of(t)}", [t], best_bias(t)))
if "d_full_span" in VAL_P:
    plan.append(("extra_span_head_decoder_only", ["d_full_span"], best_bias("d_full_span")))
plan.append(("extra_ensemble", ens, BEST_O_BIAS))

rows, hashes = [], {}
for name, tags, ob in plan:
    write_submission(tags, "_tmp_sub", ob)
    folder = f"{SUB_DIR}/{name}"
    os.makedirs(folder)
    shutil.copy("artifacts/_tmp_sub.json", f"{folder}/prediction.json")
    with zipfile.ZipFile(f"{folder}/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    h = hashlib.md5(open(f"{folder}/prediction.json", "rb").read()).hexdigest()
    if h in hashes:
        print(f"  WARNING: {name} has identical predictions to {hashes[h]}")
    hashes[h] = name
    vf1 = score(avg_probs([VAL_P[t] for t in tags]), ob)[0]["macro_f1"] if all(t in VAL_P for t in tags) else float("nan")
    sp = SPEC_OF[tags[0]]
    single = len(tags) == 1
    rows.append({"folder": name, "phase1_model": sp["arch"] if single else "ensemble",
                 "technique": (tech_of(tags[0]) + ("+span head" if sp["head"] == "span" else "")) if single else "ensemble",
                 "val_macro_f1": round(vf1, 4), "o_bias": ob,
                 "test_entities": sum(len(v) for v in json.load(open(f"{folder}/prediction.json")).values())})

for f in ["artifacts/_tmp_sub.json", "artifacts/_tmp_sub.zip"]:
    if os.path.exists(f):
        os.remove(f)
display(pd.DataFrame(rows))

with zipfile.ZipFile("artifacts/all_submissions.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(SUB_DIR):
        for fn in files:
            p = os.path.join(root, fn)
            z.write(p, arcname=os.path.relpath(p, SUB_DIR))
print("\nUpload the four artifacts/submissions/official*/official*.zip files (extras are optional).")


## 13. Offset / pipeline audit (run before every submission)


In [ ]:
bad = illegal = overlap_same = 0
for did, ents in out.items():
    t = test_texts[did]
    for s, e, lab, txt in ents:
        if t[s:e] != txt: bad += 1
        if lab not in type_labels: illegal += 1
    seen = sorted((s, e, lab) for s, e, lab, _ in ents)
    for (s1, e1, l1), (s2, e2, l2) in zip(seen, seen[1:]):
        if s2 < e1 and l1 == l2: overlap_same += 1
print("slice mismatches", bad, "| illegal labels", illegal, "| same-label overlaps", overlap_same)
slens = [e[1] - e[0] for ents in out.values() for e in ents]
print("pred span char-len  n", len(slens), "mean", round(float(np.mean(slens)), 1), "p50", int(np.median(slens)), "p95", int(np.percentile(slens, 95)))
pc = Counter(e[2] for ents in out.values() for e in ents)
tc = Counter(lab for ents in golds.values() for _, _, lab in ents)
print("entities per doc (test pred vs train gold):")
print(pd.DataFrame({"test_pred": {k: round(pc[k] / len(out), 1) for k in type_labels},
                    "train_gold": {k: round(tc[k] / len(golds), 1) for k in type_labels}}).to_string())
_ = show_results()
print("BEST single:", max(VAL_SCORE, key=VAL_SCORE.get), round(max(VAL_SCORE.values()), 4), "| ensemble val F1", round(best_f, 4))


## 14. Report checklist (§8.5)

Fill from `artifacts/p2v2_results.jsonl` and the tables above (executed numbers, not guesses):

- [ ] **Four Phase-1 architectures**, each with full FT / LoRA / LORMA: decoder_only (`d_*`), efficient (`e_*`), chunk_recurrent (`c_*`), enc_dec (`ed_*`); KD student (`d_kd_student`).
- [ ] **≥3 adaptation techniques**: full FT (4 variants), LoRA (`d_lora`), **LORMA** (`d_lorma`, compulsory), knowledge distillation (`d_kd_student`).
- [ ] KD: teacher vs student parameters (`params_M`), val F1, `train_min`, `val_infer_s`, `peak_gpu_GB`.
- [ ] Decoding: argmax vs Viterbi vs O-bias (§10b). Token-level BIO (linear / BiLSTM heads) is compared with the span-boundary head `d_full_span` (BiLSTM + start/end/type heads, `BoundaryDecoder`). Window-probability averaging vs v1 union decoding explains the precision gain (v1 val P 0.28).
- [ ] Context: causal vs bidirectional, linear vs BiLSTM head, header-context variant (`d_full_bidir_lstm_ctx`), window `MAX_LEN`/`STRIDE`.
- [ ] Class imbalance: token histogram, mild class weights, O-bias sweep table.
- [ ] Boundary analysis: round-trip with / without learned boundary rules; boundary-shift counts in §11.
- [ ] Per-type F1 table (§11) and which types remain hard (APP / RESP / counsel depend on document-level role).
- [ ] PEFT vs full FT: trainable M params and memory vs F1.
- [ ] Phase-1 transfer: compare architectures under the same recipe.
- [ ] Submit the four `artifacts/submissions/official1..4_*/` zips (four different Phase-1 models; KD + LORMA + at least 3 techniques). The `extra_*` folders (span head, ensemble) are optional additional experiments: describe them in the report but they are not part of the four.

**Kaggle how-to:** GPU T4, persist output, attach Phase-2 dataset + Phase-1 output/dataset, set `QUICK=False`. If OOM lower `H["batch"]` to 2 or `MAX_LEN` to 384. Finished experiments are skipped on restart when `FORCE_RETRAIN=False`; delete `artifacts/ckpt_lner/` if you change `MAX_LEN`, the label set or the head code.
